In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/raw/bank-additional-full.csv")

df.head()

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,deposit
0,59,admin.,married,secondary,no,2343,yes,no,unknown,5,may,1042,1,-1,0,unknown,yes
1,56,admin.,married,secondary,no,45,no,no,unknown,5,may,1467,1,-1,0,unknown,yes
2,41,technician,married,secondary,no,1270,yes,no,unknown,5,may,1389,1,-1,0,unknown,yes
3,55,services,married,secondary,no,2476,yes,no,unknown,5,may,579,1,-1,0,unknown,yes
4,54,admin.,married,tertiary,no,184,no,no,unknown,5,may,673,2,-1,0,unknown,yes


Etape importante pour la suite : 
Les variables numériques pourront être standardisées si le modèle en a besoin.
Les variables catégorielles devront être encodées en valeurs numériques, notamment avec OneHotEncoder.

In [4]:
numerical_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()

print("Variables numeriques :", numerical_features)
print("\nVariables categorielles :", categorical_features)
print("\nNombre de variables numeriques :", len(numerical_features))
print("Nombre de variables categorielles :", len(categorical_features))

Variables numeriques : ['age', 'balance', 'day', 'campaign', 'pdays', 'previous']

Variables categorielles : ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'poutcome']

Nombre de variables numeriques : 6
Nombre de variables categorielles : 9


C:\Users\phube\AppData\Local\Temp\ipykernel_3668\1479358839.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X.select_dtypes(include=["object"]).columns.tolist()


In [ ]:
categorical_features = X.select_dtypes(include=["str", "object"]).columns.tolist()
for column in categorical_features:
    print(f"\n{column}")
    print(X[column].value_counts()) 


job
job
management       2566
blue-collar      1944
technician       1823
admin.           1334
services          923
retired           778
self-employed     405
student           360
unemployed        357
entrepreneur      328
housemaid         274
unknown            70
Name: count, dtype: int64

marital
marital
married     6351
single      3518
divorced    1293
Name: count, dtype: int64

education
education
secondary    5476
tertiary     3689
primary      1500
unknown       497
Name: count, dtype: int64

default
default
no     10994
yes      168
Name: count, dtype: int64

housing
housing
no     5881
yes    5281
Name: count, dtype: int64

loan
loan
no     9702
yes    1460
Name: count, dtype: int64

contact
contact
cellular     8042
unknown      2346
telephone     774
Name: count, dtype: int64

month
month
may    2824
aug    1519
jul    1514
jun    1222
nov     943
apr     923
feb     776
oct     392
jan     344
sep     319
mar     276
dec     110
Name: count, dtype: int64

poutcome
p

In [8]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train :", X_train.shape)
print("X_test :", X_test.shape)
print("\nRepartition de y_train :")
print(y_train.value_counts(normalize=True) * 100)
print("\nRepartition de y_test :")
print(y_test.value_counts(normalize=True) * 100)

X_train : (8929, 15)
X_test : (2233, 15)

Repartition de y_train :
deposit
no     52.615074
yes    47.384926
Name: proportion, dtype: float64

Repartition de y_test :
deposit
no     52.619794
yes    47.380206
Name: proportion, dtype: float64


OK pour nos variables categorieles, on va faire du One-Hot-Encoding pour transformer chaque catégorie en une colonne binaire (0 ou 1), sans imposer de relation artificielle entre les catégories.

In [11]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline

numeric_transformer = StandardScaler()

categorical_transformer = OneHotEncoder(handle_unknown="ignore")

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numerical_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

preprocessing_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
    ]
)

X_train_processed = preprocessing_pipeline.fit_transform(X_train)
X_test_processed = preprocessing_pipeline.transform(X_test)

print("Dimensions X_train apres preprocessing :", X_train_processed.shape)
print("Dimensions X_test apres preprocessing :", X_test_processed.shape)

Dimensions X_train apres preprocessing : (8929, 50)
Dimensions X_test apres preprocessing : (2233, 50)


A noter, on est passes de 15 a 50 variables mais c'est normal, c'est le OHE qui decoupe une seule variable en plusieurs pour pouvoir la rendre binaire.
On peut le verifier par nous memes pour en etre sur avec le code suivant :

In [12]:
feature_names = preprocessing_pipeline.named_steps["preprocessor"].get_feature_names_out()

print("Nombre de variables transformees :", len(feature_names))
print("\nNoms des variables transformees :")
print(feature_names)

Nombre de variables transformees : 50

Noms des variables transformees :
['num__age' 'num__balance' 'num__day' 'num__campaign' 'num__pdays'
 'num__previous' 'cat__job_admin.' 'cat__job_blue-collar'
 'cat__job_entrepreneur' 'cat__job_housemaid' 'cat__job_management'
 'cat__job_retired' 'cat__job_self-employed' 'cat__job_services'
 'cat__job_student' 'cat__job_technician' 'cat__job_unemployed'
 'cat__job_unknown' 'cat__marital_divorced' 'cat__marital_married'
 'cat__marital_single' 'cat__education_primary' 'cat__education_secondary'
 'cat__education_tertiary' 'cat__education_unknown' 'cat__default_no'
 'cat__default_yes' 'cat__housing_no' 'cat__housing_yes' 'cat__loan_no'
 'cat__loan_yes' 'cat__contact_cellular' 'cat__contact_telephone'
 'cat__contact_unknown' 'cat__month_apr' 'cat__month_aug' 'cat__month_dec'
 'cat__month_feb' 'cat__month_jan' 'cat__month_jul' 'cat__month_jun'
 'cat__month_mar' 'cat__month_may' 'cat__month_nov' 'cat__month_oct'
 'cat__month_sep' 'cat__poutcome_failure' 

Verifications importantes avant le preprocessing :

In [13]:
assert "duration" not in X.columns
assert "deposit" not in X.columns
assert X_train_processed.shape[0] == len(y_train)
assert X_test_processed.shape[0] == len(y_test)
assert X_train_processed.shape[1] == X_test_processed.shape[1]
assert X_train_processed.shape[1] == len(feature_names)

print("Toutes les verifications sont validees.")
print("Dimensions train :", X_train_processed.shape)
print("Dimensions test :", X_test_processed.shape)
print("Nombre de variables finales :", len(feature_names))

Toutes les verifications sont validees.
Dimensions train : (8929, 50)
Dimensions test : (2233, 50)
Nombre de variables finales : 50


Ok, comme indiquee, les verifications sont validees, cela marque la fin de notre etape de preprocessing, on va pouvoir maintenant passer a l'entrainement sur plusieurs modeles et les comparer entre eux